In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Downloaded to:", path)

In [ ]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
img_dir_candidates = [root for root, dirs, files in os.walk(path) if any(f.lower().endswith(".jpg") for f in files)]

print("CSV files found:", csv_candidates)
print("Image directories found:", img_dir_candidates)

In [ ]:
import pandas as pd

metadata_path = next(p for p in csv_candidates if "metadata" in p.lower())
df = pd.read_csv(metadata_path)
print(df.shape)
print(df.columns.tolist())

class_counts = df["dx"].value_counts()
print()
print(class_counts)
print()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (largest/smallest class): {imbalance_ratio:.1f}x")
majority_baseline_acc = class_counts.max() / len(df)
print(f"'Always predict {class_counts.idxmax()}' baseline accuracy: {majority_baseline_acc:.1%}")

In [ ]:
def find_image_path(image_id):
    for img_dir in img_dir_candidates:
        candidate = os.path.join(img_dir, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

df["image_path"] = df["image_id"].apply(find_image_path)

missing = df["image_path"].isna().sum()
print(f"Images not found: {missing} out of {len(df)}")
df = df.dropna(subset=["image_path"]).reset_index(drop=True)
print("Final shape:", df.shape)

In [ ]:
import numpy as np

DX_TO_FULLNAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}
DX_CODES = sorted(DX_TO_FULLNAME.keys())  # fixed order — this defines the model's output index order
CONDITIONS = [DX_TO_FULLNAME[c] for c in DX_CODES]

code_to_index = {code: i for i, code in enumerate(DX_CODES)}
df["label"] = df["dx"].map(code_to_index)

print("Class index mapping:")
for code, idx in code_to_index.items():
    print(f"  {idx}: {code} -> {DX_TO_FULLNAME[code]}")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
import tensorflow as tf

IMG_SIZE = 224  # EfficientNetB0's standard input size

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # NO /255 here — EfficientNet has rescaling built into its own first
    # layer and expects raw [0,255] pixel values. Normalizing here would
    # cause the model to double-rescale everything down to ~[0, 0.004],
    # silently wrecking training without any error.
    return image, label

def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df["image_path"].to_numpy(), train_df["label"].to_numpy(), shuffle=True)
val_ds = make_dataset(val_df["image_path"].to_numpy(), val_df["label"].to_numpy())

for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "pixel range:", imgs.numpy().min(), "-", imgs.numpy().max())

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # phase 1: frozen base

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CONDITIONS), activation="softmax")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_df["label"])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"].to_numpy())
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))

print(f"{'Condition':<50} {'Weight':>8}")
for cls_idx, weight in sorted(class_weight_dict.items()):
    print(f"{CONDITIONS[cls_idx]:<50} {weight:>8.3f}")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase1.keras", save_best_only=True, monitor="val_loss"),
]

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase2.keras", save_best_only=True, monitor="val_loss"),
]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = val_df["label"].to_numpy()
y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = (y_pred == y_true).mean()
print(f"v3 (class-weighted) accuracy: {accuracy:.1%}")
print(f"v2 (unweighted) accuracy:     80.5%")
print(f"Majority-class baseline:      {majority_baseline_acc:.1%}")
print()
print(classification_report(y_true, y_pred, target_names=CONDITIONS, zero_division=0))
print()
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

melanoma_idx = CONDITIONS.index("Melanoma")
melanoma_recall = confusion_matrix(y_true, y_pred)[melanoma_idx, melanoma_idx] / (y_true == melanoma_idx).sum()
print(f"\nMelanoma recall: {melanoma_recall:.2f} (v2 was 0.42, v1 was 0.35)")

In [ ]:
import json, os

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v3_weighted_efficientnet.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

In [ ]:
import shutil
shutil.make_archive("v3_skin_lesion_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v3_skin_lesion_artifacts.zip")